# csv読み込み

In [0]:
SELECT *
FROM csv.`/Volumes/users/yukiteru_koide/learning/sample.csv`

# jsonの読み込み

In [0]:
SELECT * 
FROM json.`/Volumes/users/yukiteru_koide/learning/sample.json`

# CUBE

In [0]:
-- サンプルデータ
WITH sales AS (
  SELECT 'East' AS region, 'A' AS product, 100 AS revenue UNION ALL
  SELECT 'East' AS region, 'B' AS product, 150 AS revenue UNION ALL
  SELECT 'West' AS region, 'A' AS product, 200 AS revenue
)
-- CUBE: region × product の全組み合わせ＋各軸の小計＋総計
SELECT
  region,
  product,
  SUM(revenue) AS total_revenue,
  GROUPING(region)  AS g_region,   -- 1=小計/総計側（NULLは集計行）
  GROUPING(product) AS g_product
FROM sales
GROUP BY CUBE(region, product)
ORDER BY region NULLS LAST, product NULLS LAST;


# ROLLUP

In [0]:
WITH sales_monthly AS (
  SELECT 2025 AS year, 1 AS quarter, 1 AS month,  50 AS revenue UNION ALL
  SELECT 2025 AS year, 1 AS quarter, 2 AS month,  70 AS revenue UNION ALL
  SELECT 2025 AS year, 2 AS quarter, 4 AS month, 120 AS revenue
)
-- ROLLUP: year > quarter > month の階層で小計～総計を生成
SELECT
  year,
  quarter,
  month,
  SUM(revenue) AS total_revenue,
  GROUPING_ID(year, quarter, month) AS gid  -- 0=明細、値が大きいほど上位サマリ
FROM sales_monthly
GROUP BY ROLLUP(year, quarter, month)
ORDER BY year NULLS LAST, quarter NULLS LAST, month NULLS LAST;


# UDF

In [0]:
-- 使う場所を合わせる
USE CATALOG users;
USE SCHEMA yukiteru_koide;

-- UC に永続関数を作成
CREATE OR REPLACE FUNCTION users.yukiteru_koide.initcap_udf(s STRING)
RETURNS STRING
LANGUAGE SQL
RETURN initcap(s);

